In [1]:
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt


In [3]:
import torch
import gpytorch
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from torch.utils.data import TensorDataset, DataLoader
from pyproj import Transformer

In [7]:
#read in dataset. 
with open('imputed_study_data.pkl', 'rb') as f:
    df=pickle.load(f)

In [3]:
my_data=df

In [6]:
#fit a time driven GP model to estimate the predictive covariance between two sensors. 

# covariate_dkl_gp.py


# -------------------------
# Utilities
# -------------------------
def encode_wind_dir_deg(wdeg):
    # wdeg: numpy array degrees [0,360)
    rad = np.deg2rad(wdeg)
    return np.stack([np.sin(rad), np.cos(rad)], axis=1)

def project_latlon_to_meters(lat, lon, epsg_from=4326, epsg_to=3857):
    # Web Mercator projection for approximate meter coordinates
    transformer = Transformer.from_crs(epsg_from, epsg_to, always_xy=True)
    xs, ys = transformer.transform(lon.tolist(), lat.tolist())
    return np.column_stack([xs, ys])

def build_input_matrix(df, include_dew=True, include_precip=True):
    # df must contain station_lat, station_lon, temperature_2m, relative_humidity_2m,
    # dew_point_2m, wind_speed_10m, wind_direction_10m, surface_pressure, precipitation
    coords = project_latlon_to_meters(df['station_lat'].values, df['station_lon'].values)
    wind_sc = encode_wind_dir_deg(df['wind_direction_10m'].values)
    cols = [
        df['temperature_2m'].values,
        df['relative_humidity_2m'].values,
        df['wind_speed_10m'].values,
        df['surface_pressure'].values,
        df['precipitation'].values,
        df['dew_point_2m'].values
    ]
    arr = np.column_stack(cols)
    arr = np.concatenate([coords, arr, wind_sc], axis=1)
    
    return arr

# -------------------------
# Feature extractor for DKL
# -------------------------
class FeatureExtractor(torch.nn.Module):
    def __init__(self, input_dim, hidden_dims=[128,64], out_dim=32):
        super().__init__()
        layers = []
        d = input_dim
        for h in hidden_dims:
            layers.append(torch.nn.Linear(d, h))
            layers.append(torch.nn.ReLU())
            d = h
        layers.append(torch.nn.Linear(d, out_dim))
        self.net = torch.nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

# -------------------------
# DKL Sparse Variational GP
# -------------------------
class DKLVariationalGP(gpytorch.models.ApproximateGP):
    def __init__(self, inducing_points, feature_extractor, out_dim):
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(inducing_points.size(0))
        variational_strategy = gpytorch.variational.VariationalStrategy(
            self, inducing_points, variational_distribution, learn_inducing_locations=True
        )
        super().__init__(variational_strategy)
        self.feature_extractor = feature_extractor
        self.mean_module = gpytorch.means.ConstantMean()
        # RBF on feature space with ARD
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel(ard_num_dims=out_dim)
        )
    def forward(self, x):
        features = self.feature_extractor(x)
        mean_x = self.mean_module(features)
        covar_x = self.covar_module(features)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# -------------------------
# Training routine
# -------------------------
def train_dkl(X_np, y_np, num_inducing=300, lr=0.01, epochs=200, batch_size=1024, device='cpu'):
    X = torch.from_numpy(X_np).float().to(device)
    y = torch.from_numpy(y_np).float().to(device)

    m = min(num_inducing, X_np.shape[0])
    km = KMeans(n_clusters=m, random_state=0).fit(X_np)
    inducing_init = torch.from_numpy(km.cluster_centers_).float().to(device)

    feat = FeatureExtractor(input_dim=X_np.shape[1], hidden_dims=[128,64], out_dim=32).to(device)
    model = DKLVariationalGP(inducing_init, feat, out_dim=32).to(device)
    likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)

    model.train()
    likelihood.train()

    optimizer = torch.optim.Adam([
        {'params': model.feature_extractor.parameters(), 'lr': lr},
        {'params': model.covar_module.parameters(), 'lr': lr},
        {'params': model.mean_module.parameters(), 'lr': lr},
        {'params': model.variational_parameters(), 'lr': lr},
        {'params': likelihood.parameters(), 'lr': lr}
    ], lr=lr)

    mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=X.shape[0])

    dataset = TensorDataset(X, y)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        epoch_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            output = model(xb)
            loss = -mll(output, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        if (epoch+1) % 20 == 0:
            print(f"Epoch {epoch+1}/{epochs} Loss {epoch_loss:.3f}")
    return model, likelihood, feat

# -------------------------
# Prediction utilities
# -------------------------
def predict_mean_var_cov(model, likelihood, X_query_np, device='cpu'):
    model.eval()
    likelihood.eval()
    Xq = torch.from_numpy(X_query_np).float().to(device)
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        posterior = likelihood(model(Xq))
        mean = posterior.mean.cpu().numpy()
        cov = posterior.covariance_matrix.cpu().numpy()
        var = posterior.variance.cpu().numpy()
    return mean, var, cov

def compute_edge_weights_from_cov(cov):
    # cov: full predictive covariance matrix for M points
    var = np.diag(cov)
    std = np.sqrt(var + 1e-12)
    corr = cov / (std[:,None] * std[None,:])
    # clip numerical issues
    corr = np.clip(corr, -0.999999, 0.999999)
    # Gaussian mutual information per pair
    mi = -0.5 * np.log(1.0 - corr**2)
    return corr, mi



In [12]:
# Safe end-to-end small test using your functions




# Build raw inputs and check for NaNs
X_raw = build_input_matrix(df)   # uses your project_latlon_to_meters and wind encoding
print("X_raw shape:", X_raw.shape)
if np.isnan(X_raw).any():
    nan_rows = np.unique(np.where(np.isnan(X_raw))[0])[:10]
    raise RuntimeError(f"NaNs found in X_raw. Example row indices: {nan_rows}")

y = df['value'].values
if np.isnan(y).any():
    raise RuntimeError("NaNs found in target column 'value' — please impute or drop those rows.")

# Standardize inputs -> Xs
scaler = StandardScaler().fit(X_raw)
Xs = scaler.transform(X_raw).astype(np.float32)
print("Xs shape:", Xs.shape, "dtype:", Xs.dtype)

# Small subset for quick test (safe sizes)
N_test = min(200, Xs.shape[0])
Xs_small = Xs[:N_test]
y_small = y[:N_test].astype(np.float32)

# Safe predict helper (variances only)
def predict_mean_and_var_only(model, likelihood, X_query_np, device='cpu'):
    model.eval(); likelihood.eval()
    Xq = torch.from_numpy(X_query_np).float().to(device)
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        post = likelihood(model(Xq))
        return post.mean.cpu().numpy(), post.variance.cpu().numpy()

# Train tiny model on CPU with low memory footprint
device = 'cpu'
try:
    model, likelihood, feat = train_dkl(Xs_small, y_small,
                                       num_inducing=20, lr=0.01,
                                       epochs=5, batch_size=64, device=device)
    mean, var = predict_mean_and_var_only(model, likelihood, Xs_small, device=device)
    print("Done small run. mean shape:", mean.shape, "var shape:", var.shape)
    print("Mean sample:", mean[:5])
    print("Var sample:", var[:5])
except Exception as e:
    print("Exception during training/prediction:")
    traceback.print_exc()
    # If MPS or low-level crash occurs, suggest fallback to CPU (we already used CPU)
    raise


X_raw shape: (170326, 10)
Xs shape: (170326, 10) dtype: float32
Done small run. mean shape: (200,) var shape: (200,)
Mean sample: [0.6351749  0.7533783  0.7755041  0.68189317 0.64378965]
Var sample: [1.2604125  1.1194553  0.98728293 0.96196854 0.9886692 ]


In [15]:
import numpy as np
import torch
import gpytorch
from sklearn.neighbors import NearestNeighbors
import pandas as pd
from collections import defaultdict
from math import ceil

def cov_to_corr_mi_from_block(cov_block, idx_map):
    # cov_block is numpy array for the unique points in the batch
    var = np.diag(cov_block)
    std = np.sqrt(var + 1e-12)
    corr_block = cov_block / (std[:, None] * std[None, :])
    corr_block = np.clip(corr_block, -0.999999, 0.999999)
    mi_block = -0.5 * np.log(1.0 - corr_block**2)
    return corr_block, mi_block

def topk_covariances_to_edges(model, likelihood, X_query_np,
                              k=10, device='cpu', batch_pairs=1000, max_unique=500):
    """
    Compute sparse covariances for top-k spatial neighbors and return edge DataFrame.
    - X_query_np: (M, d) standardized inputs; first two cols must be projected coords.
    - k: number of neighbors per node (excluding self).
    - batch_pairs: number of (i,j) pairs to process per GP call group.
    - max_unique: maximum unique points per batch to evaluate at once (controls memory).
    Returns pandas DataFrame with columns ['i','j','cov','corr','mi'] for i<j.
    """
    M = X_query_np.shape[0]
    coords = X_query_np[:, :2]
    nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto').fit(coords)
    distances, indices = nbrs.kneighbors(coords)
    # Build unique unordered pairs (i,j) with i<j
    pairs = []
    for i in range(M):
        for j in indices[i, 1:]:
            if i < j:
                pairs.append((i, int(j)))
            elif j < i:
                pairs.append((j, int(i)))
    # Deduplicate pairs
    pairs = sorted(set(pairs))
    if len(pairs) == 0:
        return pd.DataFrame(columns=['i','j','cov','corr','mi'])

    # Process pairs in batches, grouping by unique points per batch
    edges = []
    idx = 0
    total_pairs = len(pairs)
    while idx < total_pairs:
        # take a slice of pairs
        batch = pairs[idx: idx + batch_pairs]
        # compute unique indices in this batch
        unique_idx = sorted({u for p in batch for u in p})
        # if too many unique points, reduce batch size by splitting
        if len(unique_idx) > max_unique:
            # reduce batch_pairs proportionally
            reduce_factor = ceil(len(unique_idx) / max_unique)
            new_batch_size = max(1, batch_pairs // reduce_factor)
            batch = pairs[idx: idx + new_batch_size]
            unique_idx = sorted({u for p in batch for u in p})
        # map global index -> local index in block
        idx_map = {u: ii for ii, u in enumerate(unique_idx)}
        Xblock = X_query_np[unique_idx]
        Xb_t = torch.from_numpy(Xblock).float().to(device)
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            post = likelihood(model(Xb_t))
            cov_block = post.covariance_matrix.cpu().numpy()
        # convert to corr and mi for this block
        corr_block, mi_block = cov_to_corr_mi_from_block(cov_block, idx_map)
        # extract pair values
        for (i, j) in batch:
            ii = idx_map[i]; jj = idx_map[j]
            cov_val = cov_block[ii, jj]
            corr_val = corr_block[ii, jj]
            mi_val = mi_block[ii, jj]
            edges.append((i, j, float(cov_val), float(corr_val), float(mi_val)))
        idx += len(batch)

    # Build DataFrame
    edges_df = pd.DataFrame(edges, columns=['i','j','cov','corr','mi'])
    return edges_df


In [17]:

def build_df_t_from_observations(df, t, time_col='time', station_id_col='location_id',
                                 max_time_diff='1H', method='nearest'):
    """
    Return one row per station with covariates at time t.
    - df: full observations with time and covariates
    - t: pandas Timestamp or string parseable to Timestamp
    - max_time_diff: maximum allowed time difference for nearest lookup (e.g. '1H')
    - method: 'nearest' or 'ffill' for how to pick a value when exact t missing
    """
    t = pd.to_datetime(t)
    # ensure time is datetime and sorted
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col])
    df = df.sort_values([station_id_col, time_col])

    # For each station, find the row nearest to t
    # compute absolute time difference per row
    df['time_diff'] = (df[time_col] - t).abs()
    # pick the row with minimum time_diff per station
    idx = df.groupby(station_id_col)['time_diff'].idxmin()
    df_t = df.loc[idx].drop(columns=['time_diff']).reset_index(drop=True)

    # filter out stations where nearest time is too far
    max_td = pd.to_timedelta(max_time_diff)
    df_t = df_t[(pd.to_datetime(df_t[time_col]) - t).abs() <= max_td].copy()

    # If you prefer forward fill from last observation before t:
    if method == 'ffill':
        # pivot to time index per station and ffill then select t
        df_sorted = df.set_index(time_col).sort_index()
        # groupby station and forward fill then take last <= t
        rows = []
        for sid, g in df_sorted.groupby(station_id_col):
            g2 = g[g.index <= t].ffill().tail(1)
            if not g2.empty:
                rows.append(g2.reset_index())
        if rows:
            df_t = pd.concat(rows, ignore_index=True)
        else:
            df_t = pd.DataFrame(columns=df.columns)

    return df_t


In [18]:
df.head(4)

,sensor_id,location_id,location_name,value,time,temperature_2m,relative_humidity_2m,dew_point_2m,wind_speed_10m,wind_direction_10m,surface_pressure,precipitation,station_lat,station_lon
0,8533919,2622586,홍릉로,18.0,2024-12-01 00:00:00+00:00,-0.9,100,-0.9,1.8,37,1000.3,0.0,37.580167,127.044856
1,8533919,2622586,홍릉로,20.0,2024-12-01 01:00:00+00:00,-1.7,100,-1.7,1.4,67,1000.2,0.0,37.580167,127.044856
2,8533919,2622586,홍릉로,16.0,2024-12-01 02:00:00+00:00,-2.0,100,-2.0,1.3,82,1000.1,0.0,37.580167,127.044856
3,8533919,2622586,홍릉로,19.0,2024-12-01 03:00:00+00:00,-2.1,100,-2.1,1.4,130,1000.4,0.0,37.580167,127.044856


In [27]:
# Build daily snapshots and compute top-K sparse edges per day, saving as pickle
import pandas as pd
import numpy as np
import os
import tempfile

# Parameters - tune as needed
time_col = "time"
station_col = "location_id"
date_agg = "mean"            # options: "mean", "median", "nearest"
reference_hour = 12          # used only if date_agg == "nearest" (UTC hour)
max_time_diff = "12H"        # used for nearest fallback
k_neighbors = 12
device = "cpu"
batch_pairs = 1200
max_unique = 400
out_dir = "daily_edges"     # directory to save per-day pickle files
os.makedirs(out_dir, exist_ok=True)

# Ensure df exists and time is datetime (UTC)
df_raw = df.copy()
df_raw[time_col] = pd.to_datetime(df_raw[time_col], utc=True)

# Helper: build one daily snapshot (one row per station)
def build_daily_snapshot(df_raw, day, method="mean", time_col="time", station_col="location_id"):
    day_start = pd.Timestamp(day).tz_convert('UTC')
    day_end = day_start + pd.Timedelta(days=1)
    df_day_window = df_raw[(df_raw[time_col] >= day_start) & (df_raw[time_col] < day_end)].copy()
    if df_day_window.empty:
        return pd.DataFrame()

    cov_cols = [
        'station_lat','station_lon',
        'temperature_2m','relative_humidity_2m','dew_point_2m',
        'wind_speed_10m','wind_direction_10m','surface_pressure','precipitation','value'
    ]
    cov_cols = [c for c in cov_cols if c in df_day_window.columns]

    if method in ("mean", "median"):
        agg_fn = "mean" if method == "mean" else "median"
        grouped = df_day_window.groupby(station_col)[cov_cols].agg(agg_fn)
        df_day = grouped.reset_index()
        if 'wind_direction_10m' in df_day_window.columns and method == "mean":
            tmp = df_day_window[[station_col, 'wind_direction_10m']].copy()
            tmp['rad'] = np.deg2rad(tmp['wind_direction_10m'])
            tmp['sin'] = np.sin(tmp['rad']); tmp['cos'] = np.cos(tmp['rad'])
            circ = tmp.groupby(station_col)[['sin','cos']].mean().reset_index()
            circ['wind_direction_10m'] = (np.rad2deg(np.arctan2(circ['sin'], circ['cos'])) + 360) % 360
            circ = circ[[station_col, 'wind_direction_10m']]
            df_day = df_day.drop(columns=['wind_direction_10m'], errors='ignore').merge(circ, on=station_col, how='left')
        return df_day.reset_index(drop=True)

    elif method == "nearest":
        ref = day_start + pd.Timedelta(hours=reference_hour)
        df_day_window['time_diff'] = (df_day_window[time_col] - ref).abs()
        idx = df_day_window.groupby(station_col)['time_diff'].idxmin()
        df_day = df_day_window.loc[idx].drop(columns=['time_diff']).reset_index(drop=True)
        max_td = pd.to_timedelta(max_time_diff)
        df_day = df_day[(df_day[time_col] - ref).abs() <= max_td].copy()
        return df_day
    else:
        raise ValueError("method must be 'mean','median', or 'nearest'")

# Iterate over days in your dataset
min_time = df_raw[time_col].min().floor('D')
max_time = df_raw[time_col].max().floor('D')
days = pd.date_range(min_time, max_time, freq='D', tz='UTC')

for day in days:
    out_path = os.path.join(out_dir, f"edges_{day.strftime('%Y-%m-%d')}.pkl")
    if os.path.exists(out_path):
        print("Skipping", day.date(), "already exists")
        continue
    print("Processing day", day.date())
    df_day = build_daily_snapshot(df_raw, day, method=date_agg, time_col=time_col, station_col=station_col)
    if df_day.empty:
        print(" No data for", day.date())
        continue

    # Build GP inputs and standardize
    X_query_raw = build_input_matrix(df_day)
    if np.isnan(X_query_raw).any():
        nan_rows = np.unique(np.where(np.isnan(X_query_raw))[0])
        print(" Dropping rows with NaNs:", len(nan_rows))
        mask = ~np.isnan(X_query_raw).any(axis=1)
        df_day = df_day.loc[mask].reset_index(drop=True)
        X_query_raw = X_query_raw[mask]

    Xq_s = scaler.transform(X_query_raw).astype(np.float32)

    # Compute sparse edges
    try:
        edges_df = topk_covariances_to_edges(
            model, likelihood, Xq_s,
            k=k_neighbors, device=device,
            batch_pairs=batch_pairs, max_unique=max_unique
        )
    except Exception as e:
        print(" Error computing covariances for", day.date(), ":", str(e))
        continue

    # Map indices back to station ids and add date
    if not edges_df.empty:
        edges_df['date'] = day.date()
        edges_df['station_i'] = df_day.iloc[edges_df['i']][station_col].values
        edges_df['station_j'] = df_day.iloc[edges_df['j']][station_col].values

        # Save as pickle using atomic write
        tmp_fd, tmp_path = tempfile.mkstemp(suffix=".pkl", dir=out_dir)
        os.close(tmp_fd)
        try:
            edges_df.to_pickle(tmp_path, protocol=pickle.HIGHEST_PROTOCOL)
            os.replace(tmp_path, out_path)
            print(" Saved", out_path, "edges:", len(edges_df))
        except Exception as e:
            # cleanup temp file on error
            if os.path.exists(tmp_path):
                os.remove(tmp_path)
            print(" Failed to save pickle for", day.date(), ":", str(e))
    else:
        print(" No edges produced for", day.date())


Processing day 2024-12-01
 Saved daily_edges/edges_2024-12-01.pkl edges: 199
Processing day 2024-12-02
 Saved daily_edges/edges_2024-12-02.pkl edges: 199
Processing day 2024-12-03
 Saved daily_edges/edges_2024-12-03.pkl edges: 199
Processing day 2024-12-04
 Saved daily_edges/edges_2024-12-04.pkl edges: 199
Processing day 2024-12-05
 Saved daily_edges/edges_2024-12-05.pkl edges: 199
Processing day 2024-12-06
 Saved daily_edges/edges_2024-12-06.pkl edges: 199
Processing day 2024-12-07
 Saved daily_edges/edges_2024-12-07.pkl edges: 199
Processing day 2024-12-08
 Saved daily_edges/edges_2024-12-08.pkl edges: 199
Processing day 2024-12-09
 Saved daily_edges/edges_2024-12-09.pkl edges: 199
Processing day 2024-12-10
 Saved daily_edges/edges_2024-12-10.pkl edges: 199
Processing day 2024-12-11
 Saved daily_edges/edges_2024-12-11.pkl edges: 199
Processing day 2024-12-12
 Saved daily_edges/edges_2024-12-12.pkl edges: 199
Processing day 2024-12-13
 Saved daily_edges/edges_2024-12-13.pkl edges: 199

In [26]:
edges_df['date'].unique()

array([datetime.date(2025, 8, 31)], dtype=object)